# Análisis Exploratorio de Datos (EDA)
En este cuaderno realizaremos el análisis exploratorio de datos de nuestro conjunto de datos de viviendas de California.
Este paso es fundamental en cualquier flujo de MLOps para comprender la distribución de los datos, detectar valores atípicos y entender las correlaciones entre características.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Cargar los datos
data_path = '1553768847-housing.csv'
df = pd.read_csv(data_path)

# Mostrar las primeras filas
df.head(10)

## 1. Información General del Dataset
Revisaremos la estructura, tipos de datos y la presencia de valores nulos.

In [ ]:
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas\n")
df.info()

## 2. Estadísticas Descriptivas
Observaremos las principales métricas de nuestras variables numéricas.

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', axis=1).format('{:,.2f}')

## 3. Análisis de Valores Nulos
Identificamos los valores faltantes para decidir estrategias de imputación.

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'Nulos': null_counts, '% del Total': null_pct})
null_df = null_df[null_df['Nulos'] > 0].sort_values('Nulos', ascending=False)

if len(null_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(null_df.index, null_df['Nulos'], color='#0C5ADB')
    ax.set_xlabel('Cantidad de valores nulos')
    ax.set_title('Variables con Valores Nulos')
    for i, (v, p) in enumerate(zip(null_df['Nulos'], null_df['% del Total'])):
        ax.text(v + 2, i, f'{v} ({p}%)', va='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("✅ No hay valores nulos en el dataset")

print("\nResumen de nulos:")
print(null_df if len(null_df) > 0 else "Sin valores nulos")

## 4. Distribución de la Variable Objetivo (median_house_value)
Es importante ver cómo se distribuye el valor medio de las casas para entender la variable que queremos predecir.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma con KDE
sns.histplot(df['median_house_value'], kde=True, bins=50, color='#0C5ADB', ax=axes[0])
axes[0].set_title('Distribución del Valor de Vivienda')
axes[0].axvline(df['median_house_value'].median(), color='red', linestyle='--', label=f'Mediana: ${df["median_house_value"].median():,.0f}')
axes[0].axvline(df['median_house_value'].mean(), color='orange', linestyle='--', label=f'Media: ${df["median_house_value"].mean():,.0f}')
axes[0].legend()

# Boxplot
sns.boxplot(y=df['median_house_value'], color='#21B8D2', ax=axes[1])
axes[1].set_title('Boxplot del Valor de Vivienda')

# QQ-Plot (normalidad)
from scipy import stats
stats.probplot(df['median_house_value'], plot=axes[2])
axes[2].set_title('Q-Q Plot (Normalidad)')

plt.tight_layout()
plt.show()

print(f"📊 Estadísticas de la variable objetivo:")
print(f"  Media:    ${df['median_house_value'].mean():>12,.2f}")
print(f"  Mediana:  ${df['median_house_value'].median():>12,.2f}")
print(f"  Std:      ${df['median_house_value'].std():>12,.2f}")
print(f"  Asimetría: {df['median_house_value'].skew():>8.3f}")
print(f"  Curtosis:  {df['median_house_value'].kurtosis():>8.3f}")
print(f"  Valores en el techo ($500,001): {(df['median_house_value'] == 500001).sum()} ({(df['median_house_value'] == 500001).mean()*100:.1f}%)")

## 5. Distribución de Todas las Variables Numéricas
Visualizamos la distribución de cada feature numérica con histogramas y boxplots.

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
n = len(num_cols)
fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))

for i, col in enumerate(num_cols):
    # Histograma
    sns.histplot(df[col], kde=True, bins=40, color='#0C5ADB', ax=axes[i, 0])
    axes[i, 0].set_title(f'Distribución de {col}')
    axes[i, 0].axvline(df[col].median(), color='red', linestyle='--', alpha=0.7)

    # Boxplot
    sns.boxplot(x=df[col], color='#21B8D2', ax=axes[i, 1])
    axes[i, 1].set_title(f'Boxplot de {col}')

plt.tight_layout()
plt.show()

## 6. Análisis de Asimetría (Skewness)
La asimetría nos indica si las variables están sesgadas, lo cual puede afectar el rendimiento de algunos modelos.

In [ ]:
skewness = df.select_dtypes(include=['float64', 'int64']).skew().sort_values(ascending=False)
print("Asimetría (Skewness) de variables numéricas:\n")
for col, val in skewness.items():
    emoji = "⚠️" if abs(val) > 1 else "✅"
    print(f"  {emoji} {col:>25s}: {val:>8.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#DC3545' if abs(v) > 1 else '#0C5ADB' for v in skewness.values]
ax.barh(skewness.index, skewness.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(-1, color='red', linestyle='--', alpha=0.5, label='Límite asimetría')
ax.axvline(1, color='red', linestyle='--', alpha=0.5)
ax.set_title('Asimetría por Variable')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Matriz de Correlación
Vamos a visualizar cómo se correlacionan las variables numéricas entre sí y con la variable objetivo.

In [ ]:
plt.figure(figsize=(12, 9))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
corr_matrix = numeric_df.corr()

# Máscara triangular superior
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', fmt=".2f",
            mask=mask, vmin=-1, vmax=1, center=0,
            linewidths=0.5, square=True)
plt.title('Matriz de Correlación (Triangular Inferior)', fontsize=14)
plt.tight_layout()
plt.show()

# Correlaciones con la variable objetivo ordenadas
print("\n📊 Correlaciones con median_house_value:\n")
target_corr = corr_matrix['median_house_value'].drop('median_house_value').sort_values(ascending=False)
for col, val in target_corr.items():
    bar = "█" * int(abs(val) * 20)
    sign = "+" if val > 0 else "-"
    print(f"  {sign} {col:>25s}: {val:>7.4f}  {bar}")

## 8. Relación entre Ingreso Medio y Valor de la Vivienda
Basado en la correlación, `median_income` es el mejor predictor del valor de las viviendas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
sns.scatterplot(x='median_income', y='median_house_value', alpha=0.3,
                color='#0C5ADB', s=10, data=df, ax=axes[0])
# Línea de tendencia
z = np.polyfit(df['median_income'], df['median_house_value'], 1)
p = np.poly1d(z)
x_range = np.linspace(df['median_income'].min(), df['median_income'].max(), 100)
axes[0].plot(x_range, p(x_range), 'r--', linewidth=2, label=f'Tendencia lineal')
axes[0].set_title('Ingreso Medio vs Valor de la Vivienda')
axes[0].set_xlabel('Ingreso Medio (x $10,000)')
axes[0].set_ylabel('Valor Medio de la Vivienda ($)')
axes[0].legend()

# Hexbin (densidad)
hb = axes[1].hexbin(df['median_income'], df['median_house_value'],
                     gridsize=40, cmap='YlOrRd', mincnt=1)
plt.colorbar(hb, ax=axes[1], label='Frecuencia')
axes[1].set_title('Mapa de Densidad: Ingreso vs Valor')
axes[1].set_xlabel('Ingreso Medio (x $10,000)')
axes[1].set_ylabel('Valor Medio de la Vivienda ($)')

plt.tight_layout()
plt.show()

## 9. Análisis por Proximidad al Océano
La variable categórica `ocean_proximity` puede tener un impacto significativo en el valor de las viviendas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribución de la categoría
ocean_counts = df['ocean_proximity'].value_counts()
colors = ['#0C5ADB', '#21B8D2', '#049AAC', '#007F5B', '#00D084']
axes[0].pie(ocean_counts.values, labels=ocean_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Distribución de ocean_proximity')

# Boxplot de valor por proximidad
order = df.groupby('ocean_proximity')['median_house_value'].median().sort_values(ascending=False).index
sns.boxplot(x='ocean_proximity', y='median_house_value', data=df,
            order=order, palette='Blues_d', ax=axes[1])
axes[1].set_title('Valor por Proximidad al Océano')
axes[1].tick_params(axis='x', rotation=30)

# Violinplot
sns.violinplot(x='ocean_proximity', y='median_income', data=df,
               order=order, palette='Blues_d', ax=axes[2])
axes[2].set_title('Ingreso por Proximidad al Océano')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Tabla resumen
summary = df.groupby('ocean_proximity').agg(
    n=('median_house_value', 'count'),
    valor_medio=('median_house_value', 'mean'),
    valor_mediana=('median_house_value', 'median'),
    ingreso_medio=('median_income', 'mean'),
).sort_values('valor_medio', ascending=False)
print("\n📊 Resumen por Proximidad al Océano:")
print(summary.to_string(float_format='${:,.0f}'.format))

## 10. Distribución Geográfica
Usamos las coordenadas `latitude` y `longitude` para visualizar la distribución espacial de los datos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Mapa de precios
scatter = axes[0].scatter(df['longitude'], df['latitude'],
                          c=df['median_house_value'], cmap='YlOrRd',
                          s=df['population']/100, alpha=0.4, edgecolors='none')
plt.colorbar(scatter, ax=axes[0], label='Valor Medio ($)')
axes[0].set_title('Mapa: Valor de Viviendas en California')
axes[0].set_xlabel('Longitud')
axes[0].set_ylabel('Latitud')

# Mapa de ingreso
scatter2 = axes[1].scatter(df['longitude'], df['latitude'],
                           c=df['median_income'], cmap='viridis',
                           s=10, alpha=0.5, edgecolors='none')
plt.colorbar(scatter2, ax=axes[1], label='Ingreso Medio (x $10,000)')
axes[1].set_title('Mapa: Ingreso Medio en California')
axes[1].set_xlabel('Longitud')
axes[1].set_ylabel('Latitud')

plt.tight_layout()
plt.show()

## 11. Detección de Outliers con IQR
Identificamos valores atípicos utilizando el método del Rango Intercuartílico (IQR).

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
outlier_info = []

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100
    outlier_info.append({'Variable': col, 'Outliers': n_outliers, '% del Total': round(pct, 2),
                         'Límite Inf.': round(lower, 2), 'Límite Sup.': round(upper, 2)})

outlier_df = pd.DataFrame(outlier_info).sort_values('Outliers', ascending=False)
print("📊 Detección de Outliers (Método IQR):\n")
print(outlier_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#DC3545' if v > 5 else '#0C5ADB' for v in outlier_df['% del Total']]
ax.barh(outlier_df['Variable'], outlier_df['% del Total'], color=colors)
ax.set_xlabel('% de Outliers')
ax.set_title('Porcentaje de Outliers por Variable (IQR)')
ax.axvline(5, color='red', linestyle='--', alpha=0.7, label='Umbral 5%')
ax.legend()
plt.tight_layout()
plt.show()

## 12. Features Derivadas (Feature Engineering)
Creamos nuevas variables que pueden mejorar la capacidad predictiva de los modelos.

In [ ]:
# Crear features derivadas
df_feat = df.copy()
df_feat['rooms_per_household'] = df_feat['total_rooms'] / df_feat['households']
df_feat['bedrooms_per_room'] = df_feat['total_bedrooms'] / df_feat['total_rooms']
df_feat['population_per_household'] = df_feat['population'] / df_feat['households']

# Correlación de features derivadas con el target
new_features = ['rooms_per_household', 'bedrooms_per_room', 'population_per_household']
print("📊 Correlación de features derivadas con median_house_value:\n")
for feat in new_features:
    corr = df_feat[feat].corr(df_feat['median_house_value'])
    bar = "█" * int(abs(corr) * 30)
    print(f"  {feat:>30s}: {corr:>7.4f}  {bar}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(new_features):
    sns.scatterplot(x=feat, y='median_house_value', data=df_feat,
                    alpha=0.2, s=8, color='#0C5ADB', ax=axes[i])
    axes[i].set_title(f'{feat} vs Valor')

plt.tight_layout()
plt.show()

## 13. Conclusiones del EDA
Hallazgos clave que guiarán el modelado.

In [ ]:
conclusions = [
    ("🔑 Predictor principal", "median_income tiene la correlación más alta (r ≈ 0.69) con el valor de viviendas"),
    ("⚠️ Datos truncados", "median_house_value está truncada en $500,001 (techo artificial)"),
    ("🏖️ Ubicación importa", "Las viviendas cerca de la costa tienden a valer más (ISLAND > NEAR BAY)"),
    ("📐 Asimetría", "Variables como total_rooms, population y households tienen fuerte asimetría positiva"),
    ("🔧 Valores nulos", "total_bedrooms tiene ~207 nulos (~1%), se imputarán con la mediana"),
    ("📊 Feature Engineering", "bedrooms_per_room muestra correlación negativa útil con el valor"),
    ("🌍 Geografía", "latitude y longitude capturan patrones espaciales (zonas urbanas vs rurales)"),
    ("📏 Outliers", "Varias variables tienen outliers significativos (>5%), los modelos de árboles son robustos a esto"),
]

print("=" * 70)
print("         CONCLUSIONES DEL ANÁLISIS EXPLORATORIO")
print("=" * 70)
for emoji_title, desc in conclusions:
    print(f"\n  {emoji_title}")
    print(f"    → {desc}")
print("\n" + "=" * 70)